In [14]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch, Rectangle
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
from pymannkendall import original_test
from statsmodels.stats.multitest import multipletests

Create a Switzerland only dataset

In [3]:
df = pd.read_csv("../CWData_clean7.csv")
df_ch = df[df["Country"] == "Switzerland"]
df_ch.to_csv("../Switzerland_only.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_8136\604894376.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 42

Overview map of all observations

In [7]:
df = pd.read_csv("../Switzerland_only.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
switzerland = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona")

counts = df.groupby(["latitude", "longitude"]).size().reset_index(name="count")

gdf_points = gpd.GeoDataFrame(
    counts,
    geometry=gpd.points_from_xy(counts["longitude"], counts["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:2056")

map_df_all = switzerland.to_crs("EPSG:2056")

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# plot switzerland map
map_df_all.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# plot points
gdf_points.plot(
    ax=ax,
    color="teal",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.7,
    markersize=np.log1p(gdf_points["count"]) * 40
)

legend_counts = [1, 10, 50, 100]
legend_handles = [
    plt.scatter([], [], s=np.log1p(c) * 40, color="teal", edgecolor="black", linewidth=0.3, alpha=0.7, label=str(c))
    for c in legend_counts
]
legend = ax.legend(
    handles=legend_handles,
    title="Observations",
    loc="upper left",
    frameon=True,
    labelspacing=1.5,
)
legend.get_title().set_fontsize(15)
for text in legend.get_texts():
    text.set_fontsize(12)

ax.set_title("All Observations in Switzerland", fontsize=18)
ax.set_axis_off()

plt.savefig("../Products/Switzerland_Overview.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_8136\3235360777.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Lake_Usage, 34: Lake_Usage_Nr, 35: Lake_Access, 36: Lake_Shore_State, 37: Lake_Swimming, 38: Lake_Transparency, 39: Lake_Color, 40: Lake_Odor, 41: Lake_Shore_Vegetation, 42: Lake_Shore_Vegetat

Number of observations, number of users and average number of observations per user per canton

In [10]:
df = pd.read_csv("../Switzerland_only.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

# data
canton_freq = df["Region"].value_counts()
users_per_canton = df.groupby("Region")["created_by"].nunique().sort_values(ascending=False)
obs_per_user_canton = (canton_freq / users_per_canton).dropna()
canton_order = canton_freq.sort_values(ascending=False).index

# obs per canton
canton_obs = canton_freq[canton_order]

# users per canton
canton_users = users_per_canton[canton_order]

# avg obs per user per canton
canton_opu = obs_per_user_canton[canton_order]


# plot
fig, axes = plt.subplots(3, 1, figsize=(16, 23), sharex=True)

canton_obs.plot(kind="bar", color="teal", ax=axes[0], logy=True, fontsize=13)
axes[0].set_title("a) Number of Observations per Canton", fontsize=21, fontweight="bold", loc="left")
axes[0].set_ylabel("Number of Observations", fontsize=18)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

canton_users.plot(kind="bar", color="teal", ax=axes[1], logy=True, fontsize=13)
axes[1].set_title("b) Number of Users per Canton", fontsize=21, fontweight="bold", loc="left")
axes[1].set_ylabel("Number of Users", fontsize=18)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)

canton_opu.plot(kind="bar", color="teal", ax=axes[2], fontsize=13)
axes[2].set_title("c) Average Number of Observations per User per Canton", fontsize=21, fontweight="bold", loc="left")
axes[2].set_ylabel("Average Number of Observations per User", fontsize=18)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
axes[2].set_axisbelow(True)
axes[2].set_xlabel("Canton", fontsize=18)
axes[2].tick_params(axis="x", rotation=90)

# alternating background
for i in range(0, len(canton_obs), 2):
    for ax in axes:
        ax.axvspan(i - 0.5, i + 0.5, color="grey", alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig("../Products/Switzerland_canton_stats_combined.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_8136\4046700095.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Lake_Usage, 34: Lake_Usage_Nr, 35: Lake_Access, 36: Lake_Shore_State, 37: Lake_Swimming, 38: Lake_Transparency, 39: Lake_Color, 40: Lake_Odor, 41: Lake_Shore_Vegetation, 42: Lake_Shore_Vegetat

Total monthly observations in Switzerland

In [11]:
df = pd.read_csv("../Switzerland_only.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 9))
ax.scatter(monthly["year_month_dt"], monthly["n_obs"], color="teal", s=20, zorder=3)
ax.plot(monthly["year_month_dt"], monthly["n_obs"], color="teal", linewidth=0.8, alpha=0.4)

ax.fill_between(monthly["year_month_dt"], monthly["n_obs"], alpha=0.1, color="teal")
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.title("Total Monthly Observations in Switzerland", fontsize=20)
plt.xlabel("Time", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

plt.savefig(f"../Products/Switzerland_Lineplot_monthly_Observations.png", dpi=300, bbox_inches="tight")
plt.close()

monthly

C:\Users\yanni\AppData\Local\Temp\ipykernel_8136\2863720331.py:1: DtypeWarning: Columns (0: Plastic_PET, 1: Plastic_POSoft, 2: Plastic_POHard, 3: Plastic_PS, 4: Plastic_PSE, 5: Plastic_PMultilayer, 6: Plastic_POther, 7: Plastic_Removed, 8: Plastic_River_Stagnant, 9: Waterlevel_Physical_Unit, 10: Streamtype, 11: Swimming_Quality, 12: Drinking_Quality, 13: Naturality, 14: StreamColor, 15: Stream_Ground_Visibility, 16: Stream_Animals, 17: Stream_Pollution_Reason, 18: Stream_sometimes_dry, 19: Stream_Name, 20: TempStream_snow_ice, 21: Stream_Waterquality, 22: Stream_Waterclarity, 23: StreamColor_other, 24: Stream_Vegetation, 25: Stream_Foam, 26: Stream_Algae, 27: Stream_Odor, 28: Stream_Odor_Type, 29: Stream_Odor_Type_other, 30: Stream_Litter, 31: Stream_Flow_Alteration, 32: Stream_typical_Color, 33: Lake_Usage, 34: Lake_Usage_Nr, 35: Lake_Access, 36: Lake_Shore_State, 37: Lake_Swimming, 38: Lake_Transparency, 39: Lake_Color, 40: Lake_Odor, 41: Lake_Shore_Vegetation, 42: Lake_Shore_Vegetat

,year_month,n_obs,year_month_dt
0,2017-02,23,2017-02-01
1,2017-03,37,2017-03-01
2,2017-04,29,2017-04-01
3,2017-05,49,2017-05-01
4,2017-06,59,2017-06-01
...,...,...,...
106,2025-12,82,2025-12-01
107,2026-01,95,2026-01-01
108,2026-02,126,2026-02-01
109,2026-03,234,2026-03-01


In [12]:
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

result = own.STL_decomposition(monthly, "n_obs", "Monthly Observations", "Observations")

Seasonal strength Monthly Observations: 0.652
Trend strength Monthly Observations:    0.873


In [15]:
trend = result.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} users/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.003
Sen's slope: 0.982 users/month
Tau: 0.188


In [16]:
# trend since 2019
trend_2019 = trend[trend.index >= "2019-01-01"]
mk_2019 = original_test(trend_2019.values)
print(f"Trend: {mk_2019.trend}")
print(f"p-value: {mk_2019.p:.3f}")
print(f"Sen's slope: {mk_2019.slope:.3f} users/month")
print(f"Tau: {mk_2019.Tau:.3f}")

Trend: decreasing
p-value: 0.011
Sen's slope: -1.234 users/month
Tau: -0.185


In [17]:
# trend since 2021
trend_2021 = trend[trend.index >= "2021-01-01"]
mk_2021 = original_test(trend_2021.values)
print(f"Trend: {mk_2021.trend}")
print(f"p-value: {mk_2021.p:.3f}")
print(f"Sen's slope: {mk_2021.slope:.3f} users/month")
print(f"Tau: {mk_2021.Tau:.3f}")

Trend: decreasing
p-value: 0.000
Sen's slope: -5.633 users/month
Tau: -0.753


Persistent spots Switzerland only

In [25]:
def persistent_mapper_Switzerland(filename):
    persistent = pd.read_csv(f"../Products/CSVs/{filename}.csv")
    persistent = persistent[persistent["Country"] == "Switzerland"]
    world = gpd.read_file(f"../Borders/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp", engine="fiona")

    category_colors = {
        "physical scale": "#1418fc",
        "plastic pollution": "#fbff2b",
        "soil moisture": "#82571b",
        "standing water type": "#dd6ef0",
        "stream type": "#fc1471",
        "temporary stream": "#8c14fc",
        "virtual scale": "#14c2fc"
    }

    gdf = gpd.GeoDataFrame(
        persistent,
        geometry=gpd.points_from_xy(persistent["longitude"], persistent["latitude"]),
        crs="EPSG:4326"
    ).to_crs("EPSG:2056")
    gdf = gdf.sort_values("n_users", ascending=False)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    world.to_crs("EPSG:2056").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            label=cat,
            zorder=2
        )

    # legend categories
    legend_cat_handles = [
        plt.scatter([], [], s=50, color=color, edgecolor="black", linewidth=0.3, alpha=0.8, label=cat.title())
        for cat, color in category_colors.items()
        if cat in gdf["Category"].unique()
    ]
    legend_cats = ax.legend(
        handles=legend_cat_handles,
        title="Category",
        loc="lower left",
        frameon=True,
        title_fontsize=15,
        fontsize=12
    )

    # legend user counts
    user_counts = [1, 5, 10, 20]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(u) * 70, color="lightgrey",
                    edgecolor="black", linewidth=0.3, alpha=0.8, label=str(u))
        for u in user_counts
    ]
    legend_users = ax.legend(
        handles=legend_handles,
        title="Number of Unique Users",
        title_fontsize=15,
        fontsize=12,
        loc="upper left",
        frameon=True,
    )
    ax.add_artist(legend_cats)  # show both legends

    if "14d" in filename:
        ax.set_title("Persistent Spots (Every 14 Days for ≥180 Days)", fontsize=18)
        outpath = "../Products/Switzerland_persistent_spots_map_14d.png"
    else:
        ax.set_title("Persistent Spots (Every 30 Days for ≥365 Days)", fontsize=18)
        outpath = "../Products/Switzerland_persistent_spots_map_30d.png"

    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()

In [26]:
for file in ["persistent_spots_30d", "persistent_spots_14d"]:
    persistent_mapper_Switzerland(file)

Stats on persistent spots

In [29]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]

df1_cantons = df1.groupby("Region").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_cantons = df2.groupby("Region").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_cantons.merge(df2_cantons, on="Region", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("Region", ascending=True)

df_combined

,Region,n_spots_14d,n_spots_30d
0,Bern,13,7
1,Schwyz,1,1
2,Solothurn,1,1
3,Vaud,2,2
4,Zürich,38,42


In [33]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]

df1_cantons = df1.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_cantons = df2.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_cantons.merge(df2_cantons, on="Category", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("n_spots_14d", ascending=False)

df_combined

,Category,n_spots_14d,n_spots_30d
2,temporary stream,32,39
3,virtual scale,21,12
1,soil moisture,1,1
0,physical scale,1,1


In [34]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
df1 = df1[df1["Country"] == "Switzerland"]
df2 = df2[df2["Country"] == "Switzerland"]
spots1 = df1[["latitude", "longitude"]].drop_duplicates()
spots2 = df2[["latitude", "longitude"]].drop_duplicates()
spot_counts1 = df1.groupby(["latitude", "longitude"]).size().reset_index(name="n_streaks")
spot_counts2 = df2.groupby(["latitude", "longitude"]).size().reset_index(name="n_streaks")

in_both = spots1.merge(spots2, on=["latitude", "longitude"], how="inner")

print(f"Number of identical spots: {len(in_both)}")
print(f"Set 1 number of total spots: {len(df1)}")
print(f"Set 2 number of total spots: {len(df2)}")
print(f"Set 1 total unique spots: {len(spots1)}")
print(f"Set 2 total unique spots: {len(spots2)}")
print(f"Set 1 spots that are also in set 2: {len(in_both)} ({len(in_both)/len(spots1)*100:.1f}%)")
print(f"Set 2 spots that are also in set 1: {len(in_both)} ({len(in_both)/len(spots2)*100:.1f}%)")

print(spot_counts1["n_streaks"].value_counts().sort_index())
print(spot_counts2["n_streaks"].value_counts().sort_index())

Number of identical spots: 43
Set 1 number of total spots: 55
Set 2 number of total spots: 53
Set 1 total unique spots: 43
Set 2 total unique spots: 48
Set 1 spots that are also in set 2: 43 (100.0%)
Set 2 spots that are also in set 1: 43 (89.6%)
n_streaks
1    36
2     3
3     3
4     1
Name: count, dtype: int64
n_streaks
1    43
2     5
Name: count, dtype: int64
